# 4-City Route Optimization using Genetic Algorithm

A delivery person starts at **A**, visits **B, C, and D** exactly once, and returns to **A**. The objective is to minimize total travel cost.

Instead of climbing towards a single better neighbour (like Hill Climbing), a **Genetic Algorithm (GA)** keeps a whole **population** of candidate routes and evolves them over several **generations** using **selection**, **crossover**, and **mutation**.

The program is divided into small cells so that each step can be run and explained separately.

## Example: How to enter the problem

Use the following cost matrix:

| From / To | A | B | C | D |
|---|---:|---:|---:|---:|
| **A** | 0 | 12 | 18 | 10 |
| **B** | 12 | 0 | 20 | 15 |
| **C** | 18 | 20 | 0 | 25 |
| **D** | 10 | 15 | 25 | 0 |

The starting city is **A**.

### Chromosome representation

Each **chromosome** is a route, represented as an ordered list of the cities visited between the start and the return to A. For example:

```text
['B', 'C', 'D']
```

means:

```text
A → B → C → D → A
```

### Fitness (cost) calculation

For `A → B → C → D → A`:

```text
A → B = 12
B → C = 20
C → D = 25
D → A = 10

Total = 12 + 20 + 25 + 10 = 67
```

Lower cost = fitter chromosome, since the goal is minimization.

### Example crossover

Parent 1: `B C D`
Parent 2: `D C B`

A random segment from Parent 1 is copied into the child, and the remaining cities are filled in, in the order they appear in Parent 2 (this is called **Order Crossover**, or OX). This guarantees the child is always a valid route with no repeated cities.

### Example mutation

Route: `B C D`

Swap two random positions (say B and D):

```text
D C B
```

Mutation happens with a small probability, so it only occasionally disturbs a route — this keeps some diversity in the population and helps avoid getting stuck.

In [1]:
# Cell 1: Cost matrix

def create_costs():
    return {
        'A': {'A': 0, 'B': 12, 'C': 18, 'D': 10},
        'B': {'A': 12, 'B': 0, 'C': 20, 'D': 15},
        'C': {'A': 18, 'B': 20, 'C': 0, 'D': 25},
        'D': {'A': 10, 'B': 15, 'C': 25, 'D': 0}
    }

costs = create_costs()
costs


{'A': {'A': 0, 'B': 12, 'C': 18, 'D': 10},
 'B': {'A': 12, 'B': 0, 'C': 20, 'D': 15},
 'C': {'A': 18, 'B': 20, 'C': 0, 'D': 25},
 'D': {'A': 10, 'B': 15, 'C': 25, 'D': 0}}

## Step 2: Enter the GA parameters

The next cell asks the user for the **population size** (how many routes are kept each generation) and the **number of generations** (how many times the population evolves).

In [2]:
# Cell 2: Function to take GA parameters

def take_ga_input():
    while True:
        try:
            pop_size = int(input("Enter population size: "))
            if pop_size > 0:
                break
            print("Please enter a positive number.")
        except ValueError:
            print("Please enter a whole number.")

    while True:
        try:
            generations = int(input("Enter number of generations: "))
            if generations > 0:
                break
            print("Please enter a positive number.")
        except ValueError:
            print("Please enter a whole number.")

    return pop_size, generations

pop_size, generations = take_ga_input()

print("Population size:", pop_size)
print("Number of generations:", generations)


Population size: 6
Number of generations: 8


## Step 3: Calculate the cost (fitness)

The cost function adds the cost of every journey, including the return to A. In a Genetic Algorithm this cost acts as the **fitness** of a chromosome — the lower the cost, the fitter the route.

In [3]:
# Cell 3: Route cost (fitness) function

def route_cost(route):
    full_route = ['A'] + route + ['A']
    total = 0

    for i in range(len(full_route) - 1):
        total += costs[full_route[i]][full_route[i + 1]]

    return total


## Step 4: Create the initial population

Each individual (chromosome) is a random valid ordering of B, C, and D. Together they form the starting **population**.

In [4]:
# Cell 4: Create individuals and the initial population

import random

def create_individual():
    cities = ['B', 'C', 'D']
    random.shuffle(cities)
    return cities

def create_population(pop_size):
    return [create_individual() for _ in range(pop_size)]

random.seed(12)
population = create_population(pop_size)

print("Initial Population:")
for individual in population:
    print("A → " + " → ".join(individual) + " → A",
          "| Cost:", route_cost(individual))


Initial Population:
A → B → D → C → A | Cost: 70
A → B → C → D → A | Cost: 67
A → D → C → B → A | Cost: 67
A → D → C → B → A | Cost: 67
A → B → D → C → A | Cost: 70
A → B → C → D → A | Cost: 67


## Step 5: Selection

Two random individuals are picked from the population and compete; the one with the lower cost (fitter) wins and becomes a parent. This is called **tournament selection** — it is simple and naturally favours better routes without discarding weaker ones completely.

In [5]:
# Cell 5: Tournament selection

def selection(population):
    tournament = random.sample(population, 2)
    tournament.sort(key=route_cost)
    return tournament[0]


## Step 6: Crossover

**Order Crossover (OX)** is used so that children are always valid routes (no city repeated, none missing):

1. Copy a random slice of Parent 1 directly into the child.
2. Fill the remaining positions with the cities from Parent 2, in the order they appear, skipping any city already placed.

In [6]:
# Cell 6: Order Crossover (OX)

def crossover(parent1, parent2):
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))

    child = [None] * size
    child[start:end + 1] = parent1[start:end + 1]

    fill_values = [city for city in parent2 if city not in child]

    idx = 0
    for i in range(size):
        if child[i] is None:
            child[i] = fill_values[idx]
            idx += 1

    return child


## Step 7: Mutation

With a small probability, two cities in the child route are swapped. This introduces small random changes so the population doesn't get stuck exploring the same routes.

In [7]:
# Cell 7: Mutation

def mutate(route, mutation_rate=0.2):
    route = route.copy()

    if random.random() < mutation_rate:
        i, j = random.sample(range(len(route)), 2)
        route[i], route[j] = route[j], route[i]

    return route


## Step 8: Run the Genetic Algorithm

For each generation:

1. Select two parents.
2. Apply crossover to produce a child.
3. Apply mutation to the child.
4. Repeat until the new population is full.
5. Replace the old population and track the best route seen so far.

In [8]:
# Cell 8: Genetic Algorithm main loop

def genetic_algorithm(pop_size, generations):
    population = create_population(pop_size)
    best_route = min(population, key=route_cost)
    best_cost = route_cost(best_route)

    for gen in range(generations):
        new_population = []

        for _ in range(pop_size):
            parent1 = selection(population)
            parent2 = selection(population)

            child = crossover(parent1, parent2)
            child = mutate(child)

            new_population.append(child)

        population = new_population

        current_best = min(population, key=route_cost)
        current_best_cost = route_cost(current_best)

        if current_best_cost < best_cost:
            best_route = current_best
            best_cost = current_best_cost

        print(f"Generation {gen + 1}: Best cost so far = {best_cost}")

    return best_route, best_cost

random.seed(12)
best_route, best_cost = genetic_algorithm(pop_size, generations)

print()
print("Best Route:", "A → " + " → ".join(best_route) + " → A")
print("Minimum Total Cost:", best_cost)


Generation 1: Best cost so far = 63
Generation 2: Best cost so far = 63
Generation 3: Best cost so far = 63
Generation 4: Best cost so far = 63
Generation 5: Best cost so far = 63
Generation 6: Best cost so far = 63
Generation 7: Best cost so far = 63
Generation 8: Best cost so far = 63

Best Route: A → D → B → C → A
Minimum Total Cost: 63


## Reading the solution

The program follows these steps:

```text
Initial Population
      ↓
Calculate cost (fitness) of each route
      ↓
Select parents (tournament selection)
      ↓
Crossover parents → child route
      ↓
Mutate child route
      ↓
New population complete?
   ↙              ↘
 No                Yes
 ↓                  ↓
Repeat          Track best route
                     ↓
                Next generation
                     ↓
        (until generations run out)
                     ↓
              Report best route
```

Unlike Hill Climbing, which follows a single path and stops at the first local optimum it finds, the Genetic Algorithm explores **many routes in parallel** across generations. This gives it a better chance of escaping a poor local optimum, at the cost of extra bookkeeping (population, selection, crossover, mutation).

For this small 4-city problem, the true minimum-cost route is **A → C → B → D → A** (or equivalently **A → D → B → C → A**, its reverse) with a total cost of **63**, and the Genetic Algorithm above successfully converges to it.